In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from netin.models import PATCHModel
from netin.link_formation_mechanisms import Uniform, PreferentialAttachment, TwoClassHomophily, TriadicClosure
from netin.graphs import Graph, BinaryClassNodeVector, NodeVector

from patch.constants import COLOR_MAJ, COLOR_MIN, PATH_PLOTS

In [ ]:
# EDGES = [(0,1), (0,2), (0,3), (1,2), (2,3), (2,9), (3,9), (4,5),(4,7), (4,9), (5,6), (5,7), (5,9), (6,9), (7,8), (7,9), (8,9)]
EDGES = [(0,1), (0,2), (0,5), (0,7), (0,9), (1,2), (1,4), (2,3), (2,4), (5,6), (5,7), (5,8), (6,7), (6,8), (7,8)]
H = 0.8
MIN = BinaryClassNodeVector.from_ndarray(np.asarray([0,1,1,1,1,0,0,0,0,0]))
TARGET_SELECTION = list(range(1, len(MIN) - 1))
# TARGET_SELECTION = [2,4,5,6,7,8]
TARGET_LABELS = list(range(1, len(MIN) - 1))
# TARGET_LABELS = "abcdefgh"
# SIZE_FIG=(1.73, 0.957)
SIZE_FIG=(1.065, 1.73)

# SIZE_FIG=(4 / 3, 0.957)

In [ ]:
plt.rcParams['figure.figsize'] = SIZE_FIG
plt.rcParams['font.size'] = 8


In [ ]:
g = Graph()

for node in range(len(MIN)):
    g.add_node(node)

for edge in EDGES:
    g.add_edge(*edge)

In [ ]:
h = TwoClassHomophily(homophily=H, node_class_values=MIN)
pa = PreferentialAttachment(N=len(MIN), graph=g)
u = Uniform(N=len(MIN))
tc = TriadicClosure(N=len(MIN), graph=g)

In [ ]:
p_u = u.get_target_probabilities(source=len(MIN) - 1)
p_h = h.get_target_probabilities(source=len(MIN) - 1)
p_pa = pa.get_target_probabilities(source=len(MIN) - 1)

p_pah = p_pa * p_h
p_pah /= np.sum(p_pah)

p_tc = tc.get_target_probabilities(source=len(MIN) - 1)

In [ ]:
def plot_pij(pij: NodeVector, lfm: str, color_groups: bool = False, tpl_ax: tuple = None):
    fig_exists = tpl_ax is not None
    fig, ax = plt.subplots(figsize=SIZE_FIG) if not fig_exists else tpl_ax
    ax.bar(
        range(len(TARGET_SELECTION)),
        pij[TARGET_SELECTION] / pij[TARGET_SELECTION].sum(),
        color=[COLOR_MIN if MIN[i] else COLOR_MAJ for i in TARGET_SELECTION] if color_groups else "black",
        label='Uniform'
    )

    if not fig_exists:
        ax.set_ylabel(f'$p^{{{lfm}}}_{{ij}}$', labelpad=0)
        ax.set_xlabel('target $j$', labelpad=0)
        # Set xticks and reduce spacing
        ax.set_xticks(
            ticks=range(len(TARGET_SELECTION)),
            labels=[f"${s}$" for s in TARGET_LABELS],
            # Align text baseline (not including descenders like g, y, p)
            va="baseline",
            )
        # Add more spacing between axis and tick labels to prevent overlap
        ax.tick_params(axis='x', pad=8)

    # Remove spines
    ax.spines[['top', 'right']].set_visible(False)
    # Remove y axis ticks
    ax.yaxis.set_ticks([])
    ax.set_ylim(0,.3)

    if not fig_exists:
        fig.tight_layout()
    return fig, ax

In [ ]:
f,a = plot_pij(pij=p_u, lfm="U", color_groups=False)
f.savefig(f"../{PATH_PLOTS}/pij_U.pdf")


In [ ]:
f,a = plot_pij(pij=p_h, lfm="H", color_groups=True)
f.savefig(f"../{PATH_PLOTS}/pij_H.pdf")


In [ ]:
f,a = plot_pij(pij=p_pah, lfm="PAH", color_groups=True)
f.savefig(f"../{PATH_PLOTS}/pij_PAH.pdf")


In [ ]:
f,a_ax = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(1.25*SIZE_FIG[0], SIZE_FIG[1]),
    sharex=True,
    sharey=True,
    layout="constrained",)

for p, tc, label, color, i, j in zip(
    (p_pah, p_pah, p_h, p_h, p_u),
    [False, True, False, True, True],
    ["PAH", "PAH", "H", "H", "U"],
    [True, True, True, True, False],
    [0, 0, 1, 1, 2],
    [0, 1, 0, 1, 1]
):
    p = p * p_tc if tc else p
    p = p / p.sum()  # Normalize probabilities
    plot_pij(
        pij=p,
        lfm=label,
        color_groups=color,
        tpl_ax=(f, a_ax[i,j]))

# Remove empty axes
f.delaxes(a_ax[-1, 0])

for ax, lfm in zip(a_ax[:,0], ("PAH", "H", "U")):
    ax.set_ylabel(f'$p_{{ij}}$', labelpad=0)
for ax in (a_ax[-1, -1], ):
    ax.set_xlabel('target $j$', labelpad=0)
    # Set xticks and reduce spacing
    ax.set_xticks(
        ticks=range(len(TARGET_SELECTION)),
        labels=[f"${s}$" for s in TARGET_LABELS],
        # Align text baseline (not including descenders like g, y, p)
        va="baseline",
        )
    # Add more spacing between axis and tick labels to prevent overlap
    ax.tick_params(axis='x', pad=8)
for label, ax in zip(
    ("global", "triadic closure"),
    # [.33, .66]):
    a_ax[0]):
    ax.set_title(
        # x=pos_x,
        # y=0.99,
        # s=label,
        label,
        # ha="center",
        # va="baseline",
        fontsize=8,
        # transform=f.transFigure
        )
for label, ax in zip(
    ("PAH", "H", "U"),
    (a_ax[0,0], a_ax[1,0], a_ax[-1, -1])):
    ax.text(
        x=0.05,
        y=1.,
        s=label,
        ha="left",
        va="top",
        fontsize=8,
        transform=ax.transAxes,
    )

for ax in a_ax.flatten():
    ax.set_ylim(0, .4)

# a_ax[-1,-1].set_ylabel('$p_{ij}$', labelpad=0)

f.savefig(f"../{PATH_PLOTS}/pij_ALL.pdf")

In [ ]:
f = plt.figure(figsize=(2*SIZE_FIG[0], SIZE_FIG[1]), layout="constrained")
grid = plt.GridSpec(nrows=4, ncols=3, figure=f)

ax_tc_u = f.add_subplot(grid[1, 0])
ax_tc_h = f.add_subplot(grid[2, 0], sharey=ax_tc_u)
ax_tc_pah = f.add_subplot(grid[3, 0], sharey=ax_tc_u)

ax_g_h = f.add_subplot(grid[0, 1], sharey=ax_tc_u)
ax_g_pah = f.add_subplot(grid[0, 2], sharey=ax_tc_u)

ax_v_h_u = f.add_subplot(grid[1, 1])
ax_v_pah_u = f.add_subplot(grid[1, 2])
ax_v_h_h = f.add_subplot(grid[2, 1])
ax_v_pah_pah = f.add_subplot(grid[3, 2])

l_axes_hist = [ax_g_pah, ax_tc_pah, ax_g_h, ax_tc_h, ax_tc_u]
l_axes_var = [ax_v_pah_u, ax_v_h_u, ax_v_pah_pah, ax_v_h_h]

for p, tc, label, color, ax in zip(
    (p_pah, p_pah, p_h, p_h, p_u),
    [False, True, False, True, True],
    ["PAH", "PAH", "H", "H", "U"],
    [True, True, True, True, False],
    l_axes_hist
):
    p = p * p_tc if tc else p
    p = p / p.sum()  # Normalize probabilities
    plot_pij(
        pij=p,
        lfm=label,
        color_groups=color,
        tpl_ax=(f, ax))
    ax.text(
        x=0.05,
        y=1.,
        s=label,
        ha="left",
        va="top",
        fontsize=8,
        transform=ax.transAxes,
    )

for ax, label in zip(
    l_axes_var,
    ("PAH,U", "H,U", "PAH,PAH", "H,H")):
    ax.text(
        x=.5,
        y=.5,
        s=label,
        ha="center",
        va="center",
        fontsize=8,
        fontweight="bold",
        transform=ax.transAxes,)
    # Hide axes
    ax.axis('off')

for ax in l_axes_hist:
    ax.set_ylim(0, .45)

# for ax in (ax_g_h, ax_g_pah, ax_tc_pah):
    # Set xticks and reduce spacing
ax_tc_pah.set_xlabel('target $j$', labelpad=0)
ax_tc_pah.set_xticks(
    ticks=range(len(TARGET_SELECTION)),
    labels=[f"${s}$" for s in TARGET_LABELS],
    # Align text baseline (not including descenders like g, y, p)
    va="baseline",)

ax_tc_pah.tick_params(axis='x', pad=7)

for ax in l_axes_hist:
    if ax == ax_tc_pah:
        continue
    ax.set_xticks(range(len(TARGET_SELECTION)), labels=[])

for ax in [ax_tc_u, ax_tc_h, ax_tc_pah]:
    ax.set_ylabel('$p_{ij}$', labelpad=-.050)

f.text(
    x=.33, y=.9, s="$p_{ij}$", ha="left", va="baseline", fontsize=8, rotation=90
)
f.text(
    x=0.6, y=1.025, s="global", ha="left", va="baseline", fontsize=8
)
f.text(
    x=-0.025, y=0.35, s="triadic closure", ha="left", va="baseline", fontsize=8, rotation=90
)
f.savefig(f"../{PATH_PLOTS}/pij_ALL_var.pdf")